# 65장 · 의도 라우팅과 LangChain 도구 선택

의도 라우터는 질문의 유형에 맞는 도구를 고릅니다. 두 의도가 섞이거나 분류 근거가 약한 경우에는 추가 질문이 필요합니다.

401 오류 문의와 결제 문의를 규칙으로 분류하고 경로별로 허용할 도구를 선택합니다. 복합 문의의 추가 확인과 Jev를 이용한 선택 실습도 같은 판단 기준으로 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain-core 1.6.12입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 규칙 기반 의도 분류

먼저 ‘영수증’, ‘결제’ 같은 단서로 문의를 분류합니다. 결제와 기술 단서가 함께 있거나 둘 다 없으면 clarify를 돌려줍니다. 이 규칙의 결과를 뒤에서 선택할 도구와 연결합니다.


### 셀 01 · rule_route()의 단일 의도 판정

두 조건 중 어떤 조건이 참인지 확인하며 마지막 반환값을 읽으세요.


In [1]:
def rule_route(text):
    text = text.lower()
    billing = any(word in text for word in ("결제", "영수증", "청구", "환불"))
    technical = any(word in text for word in ("api", "인증", "오류", "401"))
    if billing and not technical:
        return "billing"
    if technical and not billing:
        return "technical"
    return "clarify"

print(rule_route("API 호출에 401 오류가 납니다"))

technical


technical은 문의의 분류 결과입니다. 이 문자열을 얻은 것만으로 기술 도구가 실행되지는 않습니다.

technical은 이 규칙이 고른 처리 경로입니다. 기술 문의로 분류했다는 사실만으로 해당 도구를 실행할 권한이 생기지는 않습니다. 분류 결과를 받은 쪽에서 경로별 허용 도구를 확인하고, 의도가 겹칠 때는 추가 질문을 보냅니다.


### 분류 결과는 다음 단계의 입력

<p>rule_route의 반환값은 billing, technical, clarify 중 한 문자열입니다. 기술 문서를 실제로 읽거나 답변을 제공하는 일은 뒤의 분기와 도구 코드가 맡습니다.</p>


### 과제 02 · 복합 의도의 추가 확인

‘영수증 조회 중 401 오류가 납니다’를 기존 함수에 넣고 어디로 가는지 확인하세요.


In [2]:
# 복합 문의를 rule_route에 넣으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(rule_route(&#x27;영수증 조회 중 401 오류가 납니다&#x27;))</code></pre>
<p>실행 결과</p><pre><code>clarify</code></pre>
<p>clarify입니다. ‘영수증’은 billing을, ‘401’은 technical을 참으로 만듭니다. 두 경로 중 하나만 참이어야 반환하는 조건이어서 마지막 줄로 갑니다.</p>
</details>


### 단서가 섞이면 추가 질문

<p>결제 단서만 있으면 billing이고 기술 단서만 있으면 technical입니다. 두 단서가 함께 있거나 둘 다 없으면 clarify가 됩니다. 단지 먼저 발견된 단어로 결정하지 않습니다.</p>


## 분류별 허용 도구와 추가 확인

LangChain 도구는 모델에 제시할 수 있는 설명과 실제 호출 함수를 함께 갖습니다. 여기서는 분류의 영향을 보기 위해 모델의 자유 선택을 생략하고, 허용 목록에서 얻은 도구를 직접 invoke합니다.


### 셀 03 · 의도별 허용 도구

도구 이름과 실제 안내를 확인합니다.

복합 문의의 분류값 clarify에는 바로 실행할 도구가 없습니다. 이 값이면 도구 목록을 찾기 전에 질문을 돌려주세요. 아래의 기술 문의만 technical 목록에서 함수를 골라 실행합니다.


In [3]:
from langchain_core.tools import tool

@tool
def technical_faq(query: str) -> str:
    """API 인증 오류의 설정 점검 절차를 안내한다."""
    return "401 오류는 키 설정과 실행 환경을 확인하세요."

question = "API 인증 오류가 납니다."
allowed = {"technical": [technical_faq]}
route = rule_route(question)
if route == "clarify":
    print("결제 문의인가요, 기술 문의인가요?")
elif route in allowed:
    selected = allowed[route][0]
    print(selected.name)
    print(selected.invoke({"query": question}))
else:
    print("이 예제에는 해당 분야의 도구가 없습니다.")

technical_faq
401 오류는 키 설정과 실행 환경을 확인하세요.


technical_faq만 선택됐습니다. 안내는 수업용 함수가 반환한 문구입니다. 계정의 키를 읽거나 인증 상태를 조회한 일은 없습니다.


### 선택한 도구를 실행하는 위치

<p>allowed에는 technical_faq가 technical 경로에만 등록되어 있습니다. route가 그 키와 일치할 때 selected.invoke를 호출합니다. 이 실습에서는 Python의 조건문이 도구를 선택합니다.</p>


## 선택 실습 · Jev 분류 결과의 수용 조건

Jev 선택 2H에서는 같은 문의와 세 분류 기준을 TypeSafeClassifier.invoke에 전달합니다. 반환된 choice와 해당 선택의 probability, confidence를 본 뒤 예제의 두 기준을 적용합니다. 아래 경계값 과제의 숫자는 수업용 입력입니다. 기본 실행은 실제 모델의 확률이나 정확도를 측정하지 않습니다. Jev 계정 없이도 같은 경계값 실습을 할 수 있습니다.


### 셀 04 · 선택 실습 · Jev 분류 호출

기본값에서는 네트워크를 호출하지 않습니다. 강사가 수업 커널에 langchain-typesafe와 키를 준비한 경우에만 한 건을 요청하세요. 키 값은 출력하지 않습니다.


In [4]:
run_jev = False
if run_jev:
    import os
    from langchain_core.decisions import Choice
    from langchain_typesafe import TypeSafeClassifier
    if not os.getenv("TYPESAFE_API_KEY"):
        raise RuntimeError("수업용 Jev 키 설정을 확인하세요.")
    classifier = TypeSafeClassifier(model=os.getenv("JEV_MODEL", "jev-latest"), timeout=30)
    criteria = {"billing": "결제 내역과 영수증 문의",
                "technical": "API 인증과 프로그램 실행 오류",
                "clarify": "두 분야가 섞였거나 의도가 불분명한 문의"}
    response = classifier.invoke({"state": "API 인증 오류가 납니다.",
                                  "questions": {"route": Choice(instructions="문의 주제를 분류하세요.", criteria=criteria)}})
    result = response.choices["route"]
    print(result.choice, result.probabilities.get(result.choice, 0.0), result.confidence)
else:
    print("Jev 실제 호출을 선택하지 않았습니다.")

Jev 실제 호출을 선택하지 않았습니다.


기본 실행에서는 호출을 건너뜁니다. 실제로 요청한 경우에는 선택값, 그 선택의 확률, confidence가 나옵니다. 다음 과제의 0.80과 0.49는 경계 판단을 연습할 예시이며 이 요청의 실측값이 아닙니다.


### 선택 실습의 실제 호출 조건

<p>기본값 run_jev=False에서는 분류 서비스를 호출하지 않습니다. True로 실행해 응답을 받은 경우에만 선택지, 해당 선택지의 확률, confidence를 실제 반환값으로 읽습니다.</p>


### 과제 05 · 확신도가 낮은 결과 처리

분류값이 billing 또는 technical이고 probability가 0.75 이상, confidence가 0.50 이상일 때만 선택을 수용하세요.


In [5]:
choice, probability, confidence = 'technical', 0.80, 0.49
# 수업용 기준을 적용해 선택을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>choice, probability, confidence = &#x27;technical&#x27;, 0.80, 0.49
accepted = choice in {&#x27;billing&#x27;, &#x27;technical&#x27;} and probability &gt;= 0.75 and confidence &gt;= 0.50
print(choice if accepted else &#x27;clarify&#x27;)</code></pre>
<p>실행 결과</p><pre><code>clarify</code></pre>
<p>확률 0.80은 기준을 넘지만 confidence 0.49가 부족해 clarify입니다. 실제 운영 문턱은 평가 자료로 따로 정해야 합니다. 실제 분류까지 비교할 때는 수업용 키가 설정된 커널에서 Jev 셀의 run_jev를 True로 바꾸세요. 같은 문의에서 받은 분류와 직접 잰 응답 시간을 기록하세요. 기본 실행의 규칙 결과에서는 그 두 칸을 비워 둡니다.</p>
</details>


### 분류가 맞아 보여도 확신도가 낮을 때

<p>과제는 technical과 함께 probability 0.80, confidence 0.49를 제공합니다. 선택지 문자열 하나로 도구를 바로 부르지 말고, 주어진 수용 조건을 둘 다 만족하는지 확인합니다.</p>


## 복습과 실습 점검

### 과제 06 · 도구가 등록되지 않은 결제 경로 처리하기

결제 단서만 있는 문의를 분류하고 해당 경로에 등록된 도구 이름을 출력하세요. 해당 경로의 도구 목록이 비어 있으면 호출을 멈추세요.


In [ ]:
question = "영수증을 다시 받고 싶습니다."
# rule_route와 allowed로 분류 결과와 허용 도구 이름 목록을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>question = &quot;영수증을 다시 받고 싶습니다.&quot;
route = rule_route(question)
print(route, [item.name for item in allowed.get(route, [])])</code></pre>
<p>실행 결과</p><pre><code>billing []</code></pre>
<p>분류기는 결제 경로를 골랐지만 이 수업의 allowed에는 technical만 등록했습니다. 응답 화면은 결제 지원 경로를 안내할 수 있습니다. 분류 결과만 보고 첫 번째 기술 도구를 실행하면 다른 업무의 답이 나옵니다.</p>
</details>
